# 30. Ders: Büyük Final - Kendi Yapay Zeka Şirketini Kur (Gradio Blocks)

## Adım 1: Teorik Çerçeve ve Kariyer Vizyonu

30 hafta boyunca süren bu efsanevi yolculukta neler başardınız bir bakalım: Değişkenleri ve döngüleri kavradınız. Veri bilimine girip Pandas ile tabloları eğdiniz büktünüz. Makine öğrenmesinin beyni olan Scikit-Learn ile karar ağaçları kurdunuz. Doğal Dil İşleme (NLP) ile yapay zekaya okumayı ve duymayı, Bilgisayarlı Görme (CV) ile dünyayı görmeyi, Üretken Yapay Zeka (Generative AI) ile yoktan resim ve şiir yaratmayı öğrettiniz.

Gerçek hayatta şirketler (Google, OpenAI, Meta) sadece tek bir fonksiyon satmazlar. Müşterilere bir platform, bir 'Portal' sunarlar. Bu son dersimizde Gradio'nun en gelişmiş özelliği olan `Blocks` mimarisini kullanarak, bugüne kadar yazdığınız MoodJar ve Yılın Şiiri projelerini sekmeli (Tab) tek bir devasa Web Sitesi (Kendi Yapay Zeka şirketinizin portfolyosu) altında birleştireceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- Portfolio (Portfolyo): Bir mühendisin (veya şirketin) yapabildiği yetenekleri tek bir yerde sergilediği şov vitrini.
- Full-Stack AI Engineer (Tam Yığın YZ Mühendisi): Hem arka plandaki yapay zeka modellerini eğiten hem de onu web sitesine dönüştürüp müşteriye sunan (Uçtan uca ürün çıkaran) elit yazılımcı.
- Gradio Blocks: `gr.Interface`'in gelişmiş, profesyonel abisi. Sayfayı satır satır, sekme sekme istediğiniz gibi (Lego gibi) tasarlamanızı sağlar.
- Tab (Sekme): Aynı pencere içinde farklı sayfalara geçiş yapmanızı sağlayan butonlar.

## Adım 3: İki Farklı Yapay Zeka Modelini (İşçiyi) İşe Almak

Şirketimizin iki farklı departmanı (ürünü) olacak. Bu yüzden iki modeli de aynı anda hafızaya (RAM) alıyoruz. (İkisi de hafif olduğu için sorun olmayacaktır).

In [ ]:
import gradio as gr
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

# 1. Model: Duygu Analisti (MoodJar İçin - Türkçe)
duygu_modeli = pipeline("sentiment-analysis", model="savasy/bert-base-turkish-sentiment-cased")

# 2. Model: Şair/Yazar (Yılın Şiiri İçin - İngilizce)
yazar_model = pipeline("text-generation", model="gpt2")

print("AI Şirketimizin çalışanları (Modeller) masalarına oturdu.")

## Adım 4: Departmanların Arka Uç (Backend) Fonksiyonları

İki farklı projemizin çekirdek fonksiyonlarını tanımlıyoruz.

In [ ]:
import random

# MoodJar Fonksiyonu
def moodjar_analiz(metin):
    if len(metin.strip()) < 3:
        return "Lütfen anlamlı bir şeyler yazın."
    analiz = duygu_modeli(metin)[0]
    if analiz['label'] == "positive":
        return f"[Pozitif - %{analiz['score']*100:.1f}] Harika bir enerji! Günün hep böyle geçsin. ☀️"
    else:
        return f"[Negatif - %{analiz['score']*100:.1f}] Zor bir an. Derin nefes al, hepsi geçecek. 🌧️"

# Şiir Makinesi Fonksiyonu
def siir_yaz(kelime1, kelime2):
    prompt = f"Write a short poem about {kelime1} and {kelime2}:\n"
    sonuc = yazar_model(prompt, max_new_tokens=30, temperature=0.7, pad_token_id=50256)
    # Basitçe promptu temizle
    ham = sonuc[0]['generated_text']
    return ham.replace(prompt, "").strip()

## Adım 5: Gradio Blocks ile Sekmeli (Çoklu) Tasarım Kurma

İşte sihir burada başlıyor. `with gr.Blocks() as sirket_app:` yapısı kullanarak, sanki bir bina inşa eder gibi kodları hiyerarşik olarak yazacağız.

In [ ]:
# Blocks mimarisini başlatıyoruz (Tema ekleyerek şık bir görünüm elde edebiliriz)
with gr.Blocks(theme=gr.themes.Soft()) as sirket_app:
    
    # Şirketin Ana Başlığı
    gr.Markdown("# 🚀 GCA Yapay Zeka Teknolojileri (AI Studio)")
    gr.Markdown("Hoş geldiniz. Sol veya sağ sekmeye tıklayarak farklı yapay zeka ürünlerimizi deneyimleyebilirsiniz.")
    
    # SEKMELERİ (TABS) OLUŞTURMA
    with gr.Tabs():
        
        # --- BİRİNCİ SEKME: MOODJAR ---
        with gr.TabItem("🧠 MoodJar (His Kavanozu)"):
            gr.Markdown("### Türkçe Duygu Analizi ve Empati Asistanı")
            # Bileşenleri tanımlıyoruz
            moodjar_girdi = gr.Textbox(lines=2, label="Bugün nasılsın?")
            moodjar_buton = gr.Button("Analiz Et", variant="primary") # Mavi renkli ana buton
            moodjar_cikti = gr.Textbox(label="Yapay Zeka Cevabı")
            
            # Butona basıldığında hangi fonksiyon çalışacak, nereden alıp nereye verecek?
            moodjar_buton.click(fn=moodjar_analiz, inputs=moodjar_girdi, outputs=moodjar_cikti)
            
        # --- İKİNCİ SEKME: YILIN ŞİİRİ ---
        with gr.TabItem("🎭 Yılın Şiiri (Şair AI)"):
            gr.Markdown("### İngilizce Kelimelerden Sanat Üretimi")
            # Bileşenleri tanımlıyoruz
            with gr.Row(): # İki kutuyu yan yana (satır) hizalamak için
                kelime_1 = gr.Textbox(label="İlham 1 (Örn: Ocean)")
                kelime_2 = gr.Textbox(label="İlham 2 (Örn: Moon)")
            
            siir_buton = gr.Button("Şiiri Yarat", variant="primary")
            siir_cikti = gr.Textbox(lines=5, label="Yazılan Şiir")
            
            # Bağlantıyı kuruyoruz
            siir_buton.click(fn=siir_yaz, inputs=[kelime_1, kelime_2], outputs=siir_cikti)
            
print("Şirket binası (Arayüz tasarımı) başarıyla inşa edildi.")

## Adım 6: Şirketi Halka Açma (Launch)

Şimdi devasa projemizi (İçinde 2 farklı model barındıran tam teşekküllü web uygulamamızı) başlatalım.

In [ ]:
# Gerçek hayatta bu kodlar Colab'de değil, AWS veya Hugging Face Spaces gibi 
# kiralık bulut sunucularında 7/24 çalışacak şekilde (Deployment) ayarlanır.
sirket_app.launch(share=True)

## Adım 7: Dene ve Keşfet - Web Sitenizin Keyfini Çıkarın

Uygulamanın linkine girin. Yukarıdaki iki farklı sekmeye tıklayarak sanki profesyonel bir SaaS (Hizmet Olarak Yazılım) ürünü kullanıyormuş gibi hissedeceksiniz. Kendi ellerinizle yazdığınız bu zeka, artık bir 'Ürüne' dönüştü.

## Adım 8: Hackathon Kültürü ve Geleceğin Meslekleri Üzerine (Kariyer Tavsiyeleri)

GCA Coding Jam gibi etkinlikler (Hackathonlar) sadece kod yazılan yerler değildir. Orada sorun çözen insanlar bulunur. Yarın öbür gün üniversiteye veya bir şirkete gittiğinizde, bu 30 derste öğrendiğiniz vizyon sayesinde şu felsefeye sahip olacaksınız:

"Ben sadece Python kodu yazmam. Ben dünyadaki bir problemi görürüm, açık kaynaklı zekaları (Hugging Face) alırım, veriyi işlerim (Pandas/Sklearn) ve bunu herkesin kullanabileceği bir ürün haline getiririm (Gradio)."

Bu tanım, dünyadaki en çok aranan ve en yüksek maaşlı 'AI Product Engineer' (Yapay Zeka Ürün Mühendisi) tanımıdır.

## Adım 9: Sıra Sende Görevi (MEZUNİYET PROJESİ)

Artık sınır yok! Bu eğitim serisini bitiren bir 'AI Developer' olarak son göreviniz (Mezuniyet Projeniz) şudur:

Bugüne kadar öğrendiğiniz (Döngüler, Karar Yapıları, Pandas, Makine Öğrenmesi, Doğal Dil İşleme, Görüntü İşleme, Gradio Arayüz) özelliklerden en az 2 veya 3 tanesini BİRLEŞTİREREK dünyada hiç kimsenin yapmadığı, kendi hayal gücünüzle tasarladığınız yepyeni bir yapay zeka problemi çözün.

Fikirler:
- Kullanıcının yüklediği resimdeki yüzleri sayıp, o kişi sayısına uygun bir şiir yazan bot.
- Hastaların ateş/nabız değerlerini Pandas'tan okuyup if-else ile doktorlara uyarı veren bir web arayüzü.
- Yazılan toksik mesajları tamamen sevimli kelimelere çevirip (Text generation ile değiştirip) öyle ekrana basan melek bekçi.

Yeni ve boş bir Jupyter Notebook açın, ismini 'Benim_Projem.ipynb' yapın ve kodlamaya başlayın.

Yolunuz her zaman açık, pikselleriniz net, modelleriniz hatasız olsun. Gelecek sizin ellerinizde! 🚀